<a href="https://colab.research.google.com/github/SidhPatel-27/dsa405-project/blob/main/DSA405_002_FA26_P2_smpate27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import os
import pandas as pd
import requests
import io

# Create raw data folder if it doesn't already exist
os.makedirs('data/raw', exist_ok=True) #If it already exists leave it alone and move on
CACHE_FILE = "data/raw/wiki_climate.csv"

# If file is already caught, read that; Don't scrape again
if os.path.exists(CACHE_FILE):
    print(f"Found local file; Loading raw data directly from {CACHE_FILE}")
    df_climate = pd.read_csv(CACHE_FILE)
# If file is not found, scrape it from Wiki and save
else:
    print("No local file found; Scraping from Wikipedia")
    url_temp = "https://en.wikipedia.org/wiki/List_of_countries_by_average_yearly_temperature"

    # Send custom User-Agent so Wiki knows who we are to get permission
    headers = {'User-Agent': 'NCSU Student Project - smpate27@ncsu.edu'}
    response = requests.get(url_temp, headers=headers)

    # read_html grabs all tables; Only need first one saved in df_climate
    temp_tables = pd.read_html(io.StringIO(response.text)) #StringIO for new update, Had to look up on AI
    df_climate = temp_tables[0]

    # Save exact unmodified raw data to the raw folder
    df_climate.to_csv(CACHE_FILE, index=False)
    print(f"Scrape complete; Raw data saved to {CACHE_FILE}")

# Display the first few rows to verify
display(df_climate.head())

Found local file; Loading raw data directly from data/raw/wiki_climate.csv


,Unnamed: 0,Country or region,Continent,Temperature
0,1,Burkina Faso,Africa,30.40 °C (86.72 °F)
1,2,Mali,Africa,29.21 °C (84.58 °F)
2,3,Aruba,South America,29.17 °C (84.51 °F)
3,4,Senegal,Africa,28.90 °C (84.02 °F)
4,5,Mauritania,Africa,28.82 °C (83.88 °F)


In [49]:
print("THE AUDIT\n")

# For each column in the dataframa, climate, print the name of said column
for col in df_climate.columns:
    print(f"\nCOLUMN: '{col}'")

    # Dtypes as they are loaded and as they should be
    loaded_dtype = df_climate[col].dtype
    expected_dtype = "float64" if "temperature" in col.lower() else "string/object"
    print(f"dtype as loaded: {loaded_dtype}")
    print(f"dtype as it should be: {expected_dtype}")

    # Missing values: count and rate
    missing = df_climate[col].isna().sum()
    rate = missing / len(df_climate)
    print(f"Missing values: {missing} (Rate: {rate:.1%})")

    # Distinct values for each column
    distinct = df_climate[col].nunique(dropna=False)
    print(f"Distinct values: {distinct}")

    # Min/Max/Range check for anything numeric
    if expected_dtype == "float64":
      temp_numeric = pd.to_numeric(df_climate[col], errors="coerce")
      valid_numeric = temp_numeric.dropna()
      if not valid_numeric.empty:
        min = valid_numeric.min()
        max = valid_numeric.max()
        print(f"Min value: {min}")
        print(f"Max value: {max}")
        print(f"Range: {max - min:.2f}")
      else:
        print("Min/Max/Range: Cannot compute, all Values loaded as text")

    # Full set of distinct values for anything categorical under 30 levels
    else:
      if distinct < 30:
        print(f"Factor levels: {df_climate[col].unique().tolist()}")
      else:
        print("Factor levels: Not printed(30 or more distinct values)")


THE AUDIT


COLUMN: 'Unnamed: 0'
dtype as loaded: int64
dtype as it should be: string/object
Missing values: 0 (Rate: 0.0%)
Distinct values: 235
Factor levels: Not printed(30 or more distinct values)

COLUMN: 'Country or region'
dtype as loaded: object
dtype as it should be: string/object
Missing values: 0 (Rate: 0.0%)
Distinct values: 235
Factor levels: Not printed(30 or more distinct values)

COLUMN: 'Continent'
dtype as loaded: object
dtype as it should be: string/object
Missing values: 0 (Rate: 0.0%)
Distinct values: 10
Factor levels: ['Africa', 'South America', 'Oceania', 'North America', 'Asia', 'Central America', 'Oceania and North America', 'Europe', 'Asia and Europe', 'Antarctica']

COLUMN: 'Temperature'
dtype as loaded: object
dtype as it should be: float64
Missing values: 0 (Rate: 0.0%)
Distinct values: 225
Min/Max/Range: Cannot compute, all Values loaded as text


Defects checked for but not found:


During this audit, multiple types of defects were explicitly checked for but not found. There were no obvious sentinel values(such as 999 or N/A). Inspection of the tail of the data confirmed there were no total or subtotal rows mixed in at the bottom of the table. A check of distinct values in the "Country or region" column showed there were no duplicate keys(every country is listed uniquely). Finally, the text was not garbled by wrong character encoding(the degree symbols ° formmatted correctly as seen in header). However the audit did reveal that numbers are stored as text in the Temperature column and a junk index column was imported, both of which will be addressed in the cleaning phase.

In [50]:
col_names = ["Variable name", "Type", "Units", "Factor levels", "Valid range", "Missingness", "Notes"]
rows = [
    [
        "country",
        "string",
        "N/A (Identifier)",
        "N/A (>30 levels)",
        "Valid recognized country names",
        "0 missing, would mean geography not properly recorded by Wikipedia",
        "Must standardize names (United States of America and USA) in P3 to match World Cup data."
    ],
    [
        "continent",
        "category",
        "N/A (Categorical)",
        "Africa, South America, Oceania, North America, Asia, Central America, Oceania and North America, Europe, Asia and Europe, Antarctica",
        "Earth's continents and major sub-regions",
        "0 missing, would mean geography not properly recorded by Wikipedia",
        "Contains some hybrid regions like 'Asia and Europe' for places like Russia."
    ],
    [
        "avg_temp_c",
        "float",
        "degrees in Celsius",
        "N/A (Numeric)",
        "-35.0 to 35.0",
        "0 missing, would mean the average temp for nation(s) wasnt properly recorded by Wikipedia",
        "Raw data had text like '30.40 °C (86.72 °F)'. Should represent a clean extracted celsius float."
    ],
    [
        "team",
        "string",
        "N/A (Identifier)",
        "N/A (>30 levels)",
        "The 48 qualifying 2026 World Cup teams",
        "0 missing, would mean that a team that qualfied, wasnt properly recorded by FBred",
        "From Source 1 (FBref); Will be done in P3. To be used to merge with country from climate dataframe"
    ],
    [
        "pts_per_match",
        "float",
        "points",
        "N/A (Numeric)",
        "0.0 to 3.0",
        "0 missing, would mean that a team's average points, wasnt properly recorded by FBref",
        "From Source 1 (FBref); Will be done in P3. Total points divided by matches played(3 for win, 1 for draw, 0 for loss)"
    ]
]

df_data_dict = pd.DataFrame(rows, columns=col_names)
display(df_data_dict)
# When running this code, convert df into an interactive table to view full contents


,Variable name,Type,Units,Factor levels,Valid range,Missingness,Notes
0,country,string,N/A (Identifier),N/A (>30 levels),Valid recognized country names,"0 missing, would mean geography not properly r...",Must standardize names (United States of Ameri...
1,continent,category,N/A (Categorical),"Africa, South America, Oceania, North America,...",Earth's continents and major sub-regions,"0 missing, would mean geography not properly r...",Contains some hybrid regions like 'Asia and Eu...
2,avg_temp_c,float,degrees in Celsius,N/A (Numeric),-35.0 to 35.0,"0 missing, would mean the average temp for nat...",Raw data had text like '30.40 °C (86.72 °F)'. ...
3,team,string,N/A (Identifier),N/A (>30 levels),The 48 qualifying 2026 World Cup teams,"0 missing, would mean that a team that qualfie...",From Source 1 (FBref); Will be done in P3. To ...
4,pts_per_match,float,points,N/A (Numeric),0.0 to 3.0,"0 missing, would mean that a team's average po...",From Source 1 (FBref); Will be done in P3. Tot...


In [51]:
df_clean = df_climate.copy()
df_clean = df_clean.drop(columns="Unnamed: 0")
df_clean = df_clean.rename(columns={"Country or region": "country","Continent": "continent","Temperature": "avg_temp_c"})
display(df_clean.head())

for v in sorted(df_clean.avg_temp_c.dropna().unique()):
    print(repr(v))

df_clean["avg_temp_c"] = df_clean["avg_temp_c"].str.replace('\xa0', ' ')
df_clean["avg_temp_c"] = df_clean["avg_temp_c"].str.split(' °C').str[0]

#Had to use AI to find out that I had to replace Unicode minus sign (−) with standard ASCII minus (-)
df_clean["avg_temp_c"] = df_clean["avg_temp_c"].str.replace('−', '-')

df_clean["avg_temp_c"] = df_clean["avg_temp_c"].astype(float)

for v in sorted(df_clean.avg_temp_c.dropna().unique()):
    print(repr(v))

df_clean["continent"] = df_clean["continent"].astype("category")
display(df_clean.head())
df_clean.info()

,country,continent,avg_temp_c
0,Burkina Faso,Africa,30.40 °C (86.72 °F)
1,Mali,Africa,29.21 °C (84.58 °F)
2,Aruba,South America,29.17 °C (84.51 °F)
3,Senegal,Africa,28.90 °C (84.02 °F)
4,Mauritania,Africa,28.82 °C (83.88 °F)


'1.07\xa0°C (33.93\xa0°F)'
'1.85\xa0°C (35.33\xa0°F)'
'10.02\xa0°C (50.04\xa0°F)'
'10.18\xa0°C (50.32\xa0°F)'
'10.35\xa0°C (50.63\xa0°F)'
'10.38\xa0°C (50.68\xa0°F)'
'10.46\xa0°C (50.83\xa0°F)'
'10.49\xa0°C (50.88\xa0°F)'
'10.67\xa0°C (51.21\xa0°F)'
'10.79\xa0°C (51.42\xa0°F)'
'10.89\xa0°C (51.60\xa0°F)'
'11.35\xa0°C (52.43\xa0°F)'
'11.40\xa0°C (52.52\xa0°F)'
'11.50\xa0°C (52.70\xa0°F)'
'11.65\xa0°C (52.97\xa0°F)'
'11.66\xa0°C (52.99\xa0°F)'
'11.78\xa0°C (53.20\xa0°F)'
'11.96\xa0°C (53.53\xa0°F)'
'12.09\xa0°C (53.76\xa0°F)'
'12.22\xa0°C (54.00\xa0°F)'
'12.27\xa0°C (54.09\xa0°F)'
'12.38\xa0°C (54.28\xa0°F)'
'12.44\xa0°C (54.39\xa0°F)'
'12.83\xa0°C (55.09\xa0°F)'
'12.96\xa0°C (55.33\xa0°F)'
'13.02\xa0°C (55.44\xa0°F)'
'13.04\xa0°C (55.47\xa0°F)'
'13.05\xa0°C (55.49\xa0°F)'
'13.06\xa0°C (55.51\xa0°F)'
'13.07\xa0°C (55.53\xa0°F)'
'13.17\xa0°C (55.71\xa0°F)'
'14.50\xa0°C (58.10\xa0°F)'
'15.20\xa0°C (59.36\xa0°F)'
'15.45\xa0°C (59.81\xa0°F)'
'15.85\xa0°C (60.53\xa0°F)'
'16.30\xa0°C (61.34\xa

,country,continent,avg_temp_c
0,Burkina Faso,Africa,30.40
1,Mali,Africa,29.21
2,Aruba,South America,29.17
3,Senegal,Africa,28.90
4,Mauritania,Africa,28.82


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   country     235 non-null    object  
 1   continent   235 non-null    category
 2   avg_temp_c  235 non-null    float64 
dtypes: category(1), float64(1), object(1)
memory usage: 4.4+ KB


1. Column: Unnamed: 0  
Change made: Dropped the entire column.  
Rows/cells affected: 235 cells.  
Why: It was a useless integer index; it serves no purpose for analysis.  
What is lost: Nothing; it was just a row counter.  
2. Column: Country or region  
Change made: Renamed to country.  
Rows/cells affected: 235 cells (header).  
Why: Standardizes the variable name to lowercase, making it easier to code with and matching the data dictionary.  
What is lost: Nothing.  
3. Column: Continent  
Change made: Renamed to continent.  
Rows/cells affected: 235 cells (header).  
Why: Standardizes the variable name to lowercase to match the data dictionary.  
What is lost: Nothing.  
4. Column: Temperature  
Change made: Renamed to avg_temp_c.  
Rows/cells affected: 235 cells (header).  
Why: Clarifies the exact unit of measurement (Celsius) and metric (average) required for the analysis.  
What is lost: Nothing.  
5. Column: avg_temp_c  
Change made: Replaced non-breaking spaces (\xa0) with standard spaces.  
Rows/cells affected: 235 cells.  
Why: HTML non-breaking spaces are hidden characters that prevent the .split() function from accurately finding the space before the °C delimiter.  
What is lost: The hidden HTML encoding artifact.  
6. Column: avg_temp_c  
Change made: Split the string at °C and kept only the first element.  
Rows/cells affected: 235 cells.  
Why: The column contained both Celsius and Fahrenheit values combined in a single string (30.40 °C (86.72 °F)); the analysis only requires the Celsius number.  
What is lost: The Fahrenheit conversion and the text symbols.  
7. Column: avg_temp_c  
Change made: Replaced Unicode minus sign (−) with a standard ASCII hyphen (-).  
Rows/cells affected: 4 cells (only countries below freezing).  
Why: Float converter crashes when it encounters the Unicode minus sign; it requires a standard keyboard hyphen to recognize negative numbers.  
What is lost: Nothing; the mathematical meaning is preserved.  
8. Column: avg_temp_c  
Change made: Converted dtype from string to float64.  
Rows/cells affected: 235 cells.  
Why: The analysis requires numeric calculations (min, max, mean) which cannot be performed on text strings.  
What is lost: Nothing; the underlying numeric values are preserved.  
9. Column: continent  
Change made: Converted dtype from string to category.  
Rows/cells affected: 235 cells.  
Why: The column contains a fixed set of repeating geographical regions; converting to categorical uses memory more efficiently and matches the data dictionary.  
What is lost: Nothing.  

Rows in raw file 235  
Rows removed as exact duplicates 0  
Rows removed as near-duplicates 0  
Rows removed for other reasons 0    
Rows in cleaned file 235  
  


Columns in raw file 4  
Columns dropped 1 (Dropped 'Unnamed: 0' because it was a useless integer index)  
Columns added 0  
Columns in cleaned file 3  

This dataset is a public reference table containing the average annual temperature in Celsius for 235 countries and territories. It was compiled by volunteer Wikipedia editors who aggregated various historical climatological baseline studies to provide the general public with a simple way to compare global climates.  

While useful for broad comparisons, the curators' decision to assign a single number to an entire nation severely limits the data. By choosing to calculate a spatial average, the creators completely left out regional climate variations. For example, averaging the arctic level cold of Alaska with the tropical heat of Florida gives the United States a single, misleadingly mild average temperature. Furthermore, the dataset leaves out the specific timeframes used for these averages, blurring historical baseline decades with modern warming trends.  

Because of these intentional omissions, this dataset can support high level categorization of nations into broad climate buckets (ex. historically "hot" vs. "cold" nations). However, it cannot support precise regional weather analysis, nor can it accurately represent the specific, on-the-ground playing conditions a football team might experience in a specific host city during a summer tournament.

In [52]:
print(os.path.abspath('data/raw/wiki_climate.csv'))

/content/data/raw/wiki_climate.csv
